# Notebook 10 - Arquitetura de produção

## Arquitetura de referência para produção

### Princípios

A solução deve separar dados, features, modelos, recuperação documental, regras de negócio e geração de linguagem. O score apoia priorização, mas não autoriza desconto, acordo ou medida jurídica. Toda resposta deve distinguir fatos observados, inferências, sugestão e evidências.

### Diagrama

```mermaid
flowchart LR
    A[Fontes estruturadas] --> B[Ingestão batch/stream]
    I[Interações] --> B
    D[Documentos e políticas] --> C[Pipeline documental]
    B --> L[Data lake/versionado]
    L --> F[Feature pipeline]
    F --> T[Treino e validação]
    T --> R[Registro de modelos/features]
    C --> V[Chunking e metadados]
    V --> E[Embeddings + índice lexical]
    E --> S[Vector store]
    L --> P[Feature store online/offline]
    R --> G[Deploy batch/online]
    P --> G
    S --> G
    G --> O[Orquestrador do agente]
    O --> W[Tools: cliente, score, histórico]
    O --> Q[Regras e guardrails]
    O --> H[LLM opcional]
    W --> O
    Q --> O
    H --> U[Resposta com citações]
    O --> U
    U --> M[Analista e revisão humana]
    M --> N[Feedback, métricas e auditoria]
    N --> T
    N --> C
    N --> G
```

### Ingestão e armazenamento

- **Batch**: cargas diárias de clientes, contratos, pagamentos, interações e resultados em zona bruta imutável.
- **Streaming**: eventos de pagamento, nova interação, alteração cadastral, contestação e mudança de consentimento para atualizar score e fila quando necessário.
- **Documentos**: entrada controlada por repositório, com hash, proprietário, versão, vigência, classificação, data de publicação e data de exclusão.
- **Camadas**: bruto, validado, curado e serving. Validar esquema, chaves, duplicidade, atraso, valores nulos, distribuição e data de referência antes de publicar.
- **Privacidade**: separar identificadores, dados de contato e atributos analíticos; mascarar ou tokenizar o que não for necessário para o caso de uso.

### Pipeline de features

Manter transformações em código versionado e reutilizável entre treino e inferência. O pipeline deve:

1. validar a data de referência e impedir variáveis posteriores ao evento;
2. calcular janelas de atraso, pagamentos e interações com o mesmo relógio temporal;
3. codificar categorias com artefatos registrados, tratando categorias novas;
4. imputar apenas com parâmetros aprendidos no treino;
5. publicar o mesmo contrato no feature store offline e online;
6. registrar versão, origem, fórmula, janela, proprietário e justificativa de cada feature.

Testar paridade treino/serving, leakage, nulos, outliers, estabilidade e disponibilidade. Falha de feature crítica deve impedir a recomendação ou usar fallback explícito.

### Treinamento e registro

Treinos reprodutíveis usam dataset imutável, código, configuração, ambiente, seed e versão de features. Validar por tempo e OOT, além de validação estratificada quando apropriado. Registrar parâmetros, métricas AUC/KS, calibração, estabilidade, importância, fairness, conjunto de treino, data e aprovadores.

O registry deve promover somente modelos aprovados. Cada artefato recebe versão e relação explícita com dados, features, prompt, documentos, índice e configuração. Rollback deve ser possível para o último modelo aprovado.

### Política de retreinamento

- **Calendário**: revisão mensal e retreino trimestral como ponto de partida.
- **Performance**: disparar investigação quando AUC, KS, calibração ou recall de encaminhamento cruzarem limites definidos.
- **Drift**: monitorar PSI, distribuição de features, taxa de nulos, proporção de classes, drift de score e drift de dados por região/safra.
- **Evento**: retreino extraordinário após mudança de política, produto, fonte, definição de target ou incidente.

Retreino automático pode gerar candidato, nunca promoção automática quando houver alteração de regra ou risco material.

### Deploy

- **Batch**: gerar scores, recomendações e filas em lote para operação diária; adequado para campanhas e priorização.
- **Online**: API de baixa latência para consulta do analista, usando feature store online, modelo registrado e recuperação documental filtrada.
- **Híbrido**: score batch persistido e agente online apenas para explicar, consultar histórico atualizado e recomendar próximo passo.

Aplicar canary, shadow mode, rollout gradual, limites de taxa e rollback. A decisão comercial final permanece fora do agente.

### Vector store e documentos

Usar busca híbrida lexical + vetorial, com reranking e filtros por `documento`, versão, vigência, jurisdição, produto, público e status. Indexar chunks com tamanho estável, sobreposição controlada, título, seção, código, versão e hash do documento.

Publicar índice novo de forma atômica, testar recall/precision e manter o índice anterior para rollback. Atualização deve reindexar apenas documentos alterados. Exclusão deve remover o documento do índice ativo, bloquear sua recuperação por metadado e executar rotina de deleção verificável no armazenamento de embeddings e caches. Documentos vencidos não podem sustentar nova recomendação.

### Orquestração do agente

O orquestrador recebe pergunta e identificador, valida autorização, roteia para ferramentas permitidas e monta contexto. Ferramentas mínimas: consulta cadastral/financeira, score calibrado, histórico de interações, busca documental e registro de auditoria.

Cada ferramenta deve ter schema, timeout, limite de registros, validação de argumentos, idempotência e escopo mínimo. O agente não deve manter memória livre de conversas: permitir somente memória operacional necessária, com TTL, consentimento e segregação por cliente. Não persistir segredos ou dados financeiros em prompt/log.

O LLM é opcional e substituível. Ele pode sintetizar contexto, mas não pode criar fatos, regras ou autorizações. A camada determinística aplica impedimentos, alçadas, consentimento, identidade e encaminhamento humano antes da resposta.

### Observabilidade

Monitorar com correlação por requisição, cliente tokenizado e versão:

- qualidade e completude dos dados, schema drift, nulos e freshness;
- performance, calibração, drift, AUC/KS, estabilidade e fairness do modelo;
- precision/recall de recuperação, documentos citados, vigência e falhas de citação;
- latência, tokens, custo, taxa de erro e recusas do LLM;
- seleção, argumentos, timeout, retry e erro de cada ferramenta;
- taxa de fallback, revisão humana, edição da recomendação e incidentes;
- resultados de negócio: regularização, custo por contato, reclamações e tempo operacional.

Alertas devem ter limiar, proprietário, runbook e ação definida. Logs de auditoria são imutáveis, minimizados e retidos conforme política.

### Segurança e governança

- Autenticação forte, autorização por função e por cliente, segregação entre desenvolvimento, validação e produção.
- Criptografia em trânsito e repouso; segredos em secret manager, nunca em notebook, prompt ou log.
- Menor privilégio para ferramentas, redes e storage; rotação de credenciais e trilha de acesso.
- Defesa contra prompt injection: tratar documentos e interações como dados não confiáveis, filtrar instruções, separar contexto de comandos e validar toda saída contra regras.
- Minimização, mascaramento, retenção e exclusão de dados; evitar atributos sensíveis e proxies injustificados.
- Responsáveis definidos para dados, modelo, política, RAG, segurança e operação. Toda mudança exige revisão, teste, aprovação e versão.
- Auditoria deve guardar entrada, contexto, fontes, modelo, prompt/configuração, ferramentas, saída, revisão e decisão final.

### Decisões por nível de autonomia

| Decisão | Automatizada | Assistida | Humana |
|---|---|---|---|
| Ingestão, validação e cálculo de features | Sim, com bloqueios | Revisão de alertas | Exceções de qualidade |
| Score e ordenação operacional | Sim, dentro do modelo aprovado | Analista interpreta | Mudança de política/modelo |
| Recuperação e citação documental | Sim, com filtros de vigência | Analista confere fonte | Conflito de política |
| Sugestão de canal ou próximo contato | Sim, somente canal consentido | Analista ajusta | Caso sensível ou sem canal seguro |
| Parcelamento/oferta customizada | Não efetivar; apenas sugerir elegibilidade | Analista avalia | Alçada, exceção ou comitê |
| Contestação, fraude, recuperação judicial e medida jurídica | Não | Pode organizar evidências | Obrigatoriamente especialista |
| Alteração cadastral, desconto e acordo | Não | Não aplicável sem aprovação | Responsável autorizado |

### Fallbacks

- **Modelo indisponível**: usar último modelo aprovado ou baseline claramente identificado; se não houver, priorizar revisão humana.
- **LLM indisponível**: usar template determinístico com fatos, fontes e limitações.
- **Vector store indisponível**: usar cache de índice aprovado; se não houver evidência vigente, não recomendar oferta.
- **Ferramenta indisponível**: não estimar silenciosamente; informar a ausência e encaminhar.
- **Baixa confiança ou conflito**: ação conservadora, sem oferta automática, com revisão humana.
- **Timeout/custo excedido**: resposta parcial explícita ou fallback, nunca ocultar a falha.

### Evolução contínua

Operar inicialmente em sombra, coletar feedback estruturado do analista e comparar com grupo de controle. Avaliar mudanças offline no golden set, executar testes adversariais e fazer experimento controlado apenas quando segurança e política estiverem aprovadas. Promover mudanças por pipeline versionado; atualizar modelo, features, prompts, documentos, índice e configuração como artefatos relacionados. Qualquer regressão de segurança, faithfulness ou política interrompe o rollout e aciona rollback.